# Diffusion Models

Verify the forward noising equation, then train a small network to predict sampled noise on synthetic two-dimensional data.

In [ ]:
import torch
from torch import nn

torch.manual_seed(40)
torch.set_num_threads(1)
print('CPU | diffusion noising and noise prediction')

In [ ]:
clean = torch.tensor([[2.]], dtype=torch.float64)
noise = torch.tensor([[-1.]], dtype=torch.float64)
alpha_bar = torch.tensor(0.81, dtype=torch.float64)
noisy = alpha_bar.sqrt() * clean + (1 - alpha_bar).sqrt() * noise
torch.testing.assert_close(noisy, torch.tensor([[1.3641101056459328]], dtype=torch.float64))
print('clean:', clean.item(), '| noise:', noise.item(), '| noisy:', noisy.item())

In [ ]:
data = torch.randn(256, 2) * 0.6 + torch.tensor([1., -1.])
denoiser = nn.Sequential(nn.Linear(3, 32), nn.Tanh(), nn.Linear(32, 2))
optimizer = torch.optim.Adam(denoiser.parameters(), lr=0.01)
for _ in range(80):
    clean_batch = data[torch.randint(len(data), (64,))]
    step = torch.rand(64, 1)
    noise_batch = torch.randn_like(clean_batch)
    noisy_batch = (1 - step).sqrt() * clean_batch + step.sqrt() * noise_batch
    prediction = denoiser(torch.cat([noisy_batch, step], dim=1))
    loss = nn.functional.mse_loss(prediction, noise_batch)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
assert torch.isfinite(loss)
print(f'tiny noise-prediction loss={loss.item():.4f}')